# `ChronDate`: Core Operations & Timeline Arithmetic

In [1]:
from mesomath import ChronDate as Date

## **1. Instantiation (The Four Pathways)**

A `ChronDate` instance can be constructed using four distinct chronological pathways, depending on the format of your source data:

### **Path 1: Julian Day Number (Absolute UT)**

Instantiates a date directly from a standard floating-point Julian Day (JD) number. This is the core native format of the engine.

In [2]:
date = Date(1748872.5)
print(f"Direct JD instantiation: {date}")

Direct JD instantiation: ChronDate(jd=1748872.5)


### **Path 2: Julian Calendar Date**

Constructs an instance from a historical Julian year, month, and day. It automatically handles proleptic or standard configurations depending on the period.

In [3]:
date = Date.from_julian(-378, 5, 17)  # Instantiation from Julian Calendar (-378-05-17)
print(f"Julian calendar instantiation: {date}")

Julian calendar instantiation: ChronDate(jd=1583129.5)


### **Path 3: Gregorian Calendar Date**

Constructs an instance using the modern Gregorian calendar rules, fully supporting proleptic extensions for deep historical retro-calculations.

In [4]:
date = Date.from_gregorian(2026, 5, 28)
print(f"Gregorian calendar instantiation: {date}")

Gregorian calendar instantiation: ChronDate(jd=2461188.5)


### **Path 4: Babylonian King Date (Historical Chronology)**

The most advanced entry point. It resolves a canonical Babylonian date into an absolute astronomical point by querying the internal Parker-Dubberstein database. It requires the king code, the regnal year, the historical lunar month, and the tablet day.

In [5]:
# Query available historical king codes
print(Date.kings())

Code       | King Name                 | Start Date (J)
------------------------------------------------------------
k001       | Nabopolassar              | -625/04/05
k002       | Nebuchadnezzar            | -603/04/02
k003       | Amel_Marduk               | -560/04/06
k004       | Nergal-Shar-Usur          | -558/04/14
k005       | Nabunaid                  | -554/03/31
k006       | Cyrus                     | -537/03/24
k007       | Cambyses                  | -528/04/12
k008       | Darius I                  | -520/04/14
k009       | Xerxes                    | -484/04/06
k010       | Artaxerxes I              | -463/04/13
k011       | Darius II                 | -422/04/11
k012       | Artaxerxes II             | -403/04/10
k013       | Artaxerxes III            | -357/04/12
k014       | Arses                     | -336/04/19
k015       | Darius III                | -334/03/29
k016       | Alexander III             | -329/04/03
k017       | Philip Arrahidaeus        | -321/04/04

> *Note: The `Start Date (J)` in the table above corresponds to King's regnal year 1, month 1, day 1.*

In [6]:
# Year 26 of Artaxerxes II, Month 2 (Aiaru), Day 14
date = Date.from_babylonian(king_code="k012", year=26, month=2, day=14)
print(f"Babylonian historical instantiation: {date}")

Babylonian historical instantiation: ChronDate(jd=1583129.5)


## **2. Core Operations & Timeline Arithmetic**

Once instantiated, a `ChronDate` object operates as an immutable point on the timeline. Internally, all operations are calculated using absolute precision floating-point Julian Days (UT). However, the class exposes a fluid API to extract representations across modern and ancient calendars, query dynastic contexts, and perform timeline arithmetic.

In the standard interactive terminal (`babcalc`), the class is available under the clean alias `Date`. Let us instantiate two anchors to explore these operations: an ancient tablet date from the Seleucid Era and a modern contemporary date.

In [7]:
date = Date.from_babylonian("k019", 45, 5, 19)
today = Date.from_gregorian(2026, 5, 29)

print(f"Ancient Anchor: {date}")
print(f"Modern Anchor:  {today}")

Ancient Anchor: ChronDate(jd=1624122.5)
Modern Anchor:  ChronDate(jd=2461189.5)


### **2.1. Calendar Conversions & Introspection**

`ChronDate` objects decouple the internal astronomical time placement from its civil expressions. Five core properties allow the inspection of any timeline node:

* **`__call__()`:** Invoking the instance directly (`date()`) returns its raw, continuous Julian Day number as a standard Python `float`.
* **`.julian`:** Returns a 7-element tuple representing `(year, month, day, hour, minute, second, microsecond)` in the Julian Calendar. It operates as a proleptic calendar for dates preceding its historical implementation.
* **`.gregorian`:** Returns a 7-element tuple in the standard modern Gregorian Calendar (proleptic for deep historical retro-calculations).
* **`.babylonian`:** Resolves the exact local Mesopotamian calendar parameters (Regnal Year, Lunar Month, and Tablet Day).
* **`.context`:** Queries the internal historical database to extract political, dynastic, or co-regency details matching the active date.

In [8]:
print(f"Raw Julian Day float:        {date()}")
print(f"Julian Calendar tuple:       {date.julian}")
print(f"Proleptic Gregorian tuple:   {date.gregorian}")
print(f"Babylonian Calendar string:  {date.babylonian}")
print(f"Historical Dynastic Context: {date.context}")

Raw Julian Day float:        1624122.5
Julian Calendar tuple:       (-266, 8, 10, 0, 0, 0, 0)
Proleptic Gregorian tuple:   (-266, 8, 6, 0, 0, 0, 0)
Babylonian Calendar string:  Year 45 of Seleucid Era, month: 5 (Abu), day: 19
(Dynasty): Year 46 of Seleucid Era (Continuous count starting 312/311 BC))
(Regnal): Year 15 of Antiochus I Soter (Co-regent from -291))
Historical Dynastic Context: None


### **2.2. The Proleptic Babylonian Calendar**

When calculating dates outside the strict limits of surviving historical records (such as modern dates), `MesoTimes` projects a **Proleptic Babylonian Calendar**.

In [9]:
print(f"Modern Julian representation:   {today.julian}")
print(f"Modern Gregorian representation: {today.gregorian}")
print(f"Proleptic Babylonian computation: {today.babylonian}")
print(f"Proleptic Context fallback:     {today.context}")

Modern Julian representation:   (2026, 5, 16, 0, 0, 0, 0)
Modern Gregorian representation: (2026, 5, 29, 0, 0, 0, 0)
Proleptic Babylonian computation: Year 2026 of Proleptic Babylonian Calendar, month: Simanu, day: 13
No historical context available for proleptic dates.
Proleptic Context fallback:     None


> **Mathematical Grounding of the Proleptic Engine**
> The proleptic Babylonian calendar is structurally mapped using the year-and-month distribution schema of the **Metonic Cycle** (the 19-year intercalation cycle stabilized during the late Babylonian/Seleucid era).
>
> However, a strict mathematical Metonic mapping introduces a systematic drift of approximately **two hours per cycle**. To prevent this chronological desynchronization, `MesoTimes` does not rely on a rigid cycle; instead, it delegates the start and duration of every proleptic month to the **true astronomical neomenia calculated at Babylon's local visual horizon**. This hybrid approach guarantees that the projected calendar remains perfectly in phase with actual lunar physics over thousands of years.

### **2.3. Timeline Arithmetic and Ordering**

`ChronDate` instances support standard arithmetic operators and total ordering comparisons. Because instances are immutable, shifting a date returns a completely new `ChronDate` node.

* **Interval Derivation (`date1 - date2`):** Subtracting one instance from another yields a `float` representing the absolute distance between both nodes in elapsed days.
* **Timeline Shifting (`date + int` / `date - int`):** Adding or subtracting loose integers steps the timeline forward or backward by that exact number of days.
* **Relational Operators:** Relational checks (`>`, `<`, `>=`, `<=`, `==`, `!=`) evaluate chronological positioning by comparing the absolute underlying Julian Day values.

In [10]:
print(f"Days elapsed between anchors: {today - date} days")
print(f"Shifting timeline (+17 days): {(date + 17).babylonian}")
print(f"Chronological ordering check (today > date): {today > date}")

Days elapsed between anchors: 837067.0 days
Shifting timeline (+17 days): Year 45 of Seleucid Era, month: 6 (Ululu), day: 7
Chronological ordering check (today > date): True


### **2.4. Hashing and Collection Persistence**

`ChronDate` implements secure hashing (`__hash__`). This enables instances to be safely stored within standard Python hashed collections, such as sets or as keys in dictionaries, preventing mutable side effects when managing large sets of archaeological text dates.

In [11]:
day_set = {date, (date + 17), today}
print("Iterating over a hashed set of ChronDate objects:")
for historical_date in day_set:
    print(f" -> {historical_date.babylonian}")

Iterating over a hashed set of ChronDate objects:
 -> Year 45 of Seleucid Era, month: 5 (Abu), day: 19
 -> Year 45 of Seleucid Era, month: 6 (Ululu), day: 7
 -> Year 2026 of Proleptic Babylonian Calendar, month: Simanu, day: 13


### **2.5. Macro-Calendrical Matrix Dispatchers (`bab_year_calendar`)**

To visualize the overarching lunisolar grid of any given historical or theoretical boundary, `ChronDate` exposes the `bab_year_calendar()` method. This engine acts as a dynamic dispatcher that decides, based on localized archaeological and algorithmic dataset availability, whether to render a strict empirical record or a cyclical astronomical projection.

The structural grid outputs a terminal report tracking the localized nomenclature of months (*Nisānu* through *Addaru*), their calculated mathematical duration (29 to 30 days based on lunar horizon visibility), and their alignment with historical Julian dates.

#### **Case A: The Regnal Era Matrix (Historical Empirical)**

When initialized inside well-documented historical intervals, the header dynamically updates to display the formal regnal years of the localized rulers or global overarching eras:

In [12]:
date = Date.from_julian(74, 10, 15)
print(date.bab_year_calendar())


--- HISTORIC BABYLONIAN CALENDAR: SELEUCID ERA YEAR 385 ---
#  Month Name     Start Date (Y/M/D)    JDE Start      Length
------------------------------------------------------------------------
1  Nisanu         74/04/17              1748192.5      30 days
2  Aiaru          74/05/17              1748222.5      29 days
3  Simanu         74/06/15              1748251.5      30 days
4  Duzu           74/07/15              1748281.5      30 days
5  Abu            74/08/14              1748311.5      29 days
6  Ululu          74/09/12              1748340.5      29 days
7  Tashritu       74/10/11              1748369.5      30 days
8  Arahsamnu      74/11/10              1748399.5      29 days
9  Kislimu        74/12/09              1748428.5      29 days
10 Tebetu         75/01/07              1748457.5      30 days
11 Shabatu        75/02/06              1748487.5      29 days
12 Addaru         75/03/07              1748516.5      30 days
------------------------------------------------

#### **Case B: The Imperial Era Matrix (Historical Seleucid)**

As the historical timeline advances, the underlying engine automatically switches its anchoring references to trace long-form continuous reporting frameworks such as the Seleucid accounting loops:

In [13]:
date = Date.from_julian(75, 10, 15)
print(date.bab_year_calendar())


--- HISTORIC BABYLONIAN CALENDAR: SELEUCID ERA YEAR 386 ---
#  Month Name     Start Date (Y/M/D)    JDE Start      Length
------------------------------------------------------------------------
1  Nisanu         75/04/06              1748546.5      29 days
2  Aiaru          75/05/05              1748575.5      30 days
3  Simanu         75/06/04              1748605.5      30 days
4  Duzu           75/07/04              1748635.5      30 days
5  Abu            75/08/03              1748665.5      30 days
6  Ululu          75/09/02              1748695.5      29 days
7  Tashritu       75/10/01              1748724.5      30 days
8  Arahsamnu      75/10/31              1748754.5      29 days
9  Kislimu        75/11/29              1748783.5      29 days
10 Tebetu         75/12/28              1748812.5      29 days
11 Shabatu        76/01/26              1748841.5      30 days
------------------------------------------------------------------------
Year ends on: 76/02/25 (JDE 1748871.5)

> **Chronological Boundary Artifact**
> The year 386 of the Seleucid Era (75/76 CE) marks the absolute terminal boundary of the empirical data compiled in Parker & Dubberstein (1971). Because the underlying database lacks any subsequent historical records beyond JDE 1748871.5, the engine cannot fetch the start date of the following year's *Nisānu*. 
>
> Consequently, the 12th month (*Addaru*) is omitted from the tabular grid, and both the reported **Year ends on** boundary and the **Total year duration** (325 days) are historically incomplete and mathematically truncated.

#### **Case C: The Proleptic Fallback Grid**

If a target date falls into a structural chronological gap where the empirical cuneiform database (`kingdates`) does not contain verified data points, the method downgrades gracefully. It switches to a proleptic calculation model to simulate astronomical ideal cycles based on the neomenia as contemplated from a given `city` or custom horizon coordinates (falling back to Babylon by default, see the [next notebook](ChronDate2.ipynb) for a list of Observatories):

In [14]:
date = Date.from_julian(76, 10, 15)
print(date.bab_year_calendar(city="Babylon", ziggurat=0.0))


--- PROLEPTIC BABYLONIAN CALENDAR: YEAR 76 ---
#   Month Name   Start Date (Y/M/D)   JDE Start       Length
---------------------------------------------------------------------
1   Nisanu       76/03/24             1748899.5       30     days
2   Aiaru        76/04/23             1748929.5       29     days
3   Simanu       76/05/22             1748958.5       30     days
4   Duzu         76/06/21             1748988.5       31     days
5   Abu          76/07/22             1749019.5       30     days
6   Ululu        76/08/21             1749049.5       29     days
7   Tashritu     76/09/19             1749078.5       29     days
8   Arahsamnu    76/10/18             1749107.5       29     days
9   Kislimu      76/11/16             1749136.5       30     days
10  Tebetu       76/12/16             1749166.5       29     days
11  Shabatu      77/01/14             1749195.5       29     days
12  Addaru       77/02/12             1749224.5       30     days
13  Addaru II    77/03/14    

> **Intercalary Recognition (`Addaru II` / `Ululu II`)**
> The historical grid reports empirical leap intercalations manually declared by the administration of the *Esagila* temple or royal decrees, while the proleptic framework applies mathematical cycle distributions. Intercalary months are explicitly tracked and logged in the terminal matrix output with a suffix (e.g., `Addaru II`).